# Assignment 2 — executable data preprocessing

Ruganji Prince · 04-652 Artificial Intelligence System Design

Current-stage audit, train-only preprocessing and descriptive diagnostics. Targets are nominal; source positions are metadata. No downstream predictors, optimization or performance comparisons are executed. Original diagnostic image bytes are retained. The three representations below are preprocessing policies, not selected predictive models.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/raw/pv_fault_dataset.csv').is_file())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.preprocessing.study import PreprocessingStudy
from src.preprocessing.common import load,sha,SENSORS
from src.pipeline_builder import POLICIES
study=PreprocessingStudy()


## Raw schema and data quality

Six float64 predictors accompany nominal uint8 f_nv. Zero conventional NaNs does not establish a missingness mechanism. The 62 suspicious paired zeros remain observations; negative Celsius values are not automatically invalid.

In [2]:
study.load_data()
print(study.raw.head(10).to_string(index=False))
print(study.raw.describe().to_string())
print('Missing counts and percentages:')
print(pd.DataFrame({'count':study.raw.isna().sum(),'percent':100*study.raw.isna().mean()}).to_string())
print('Complete rows:',int(study.raw.notna().all(axis=1).sum()),'; incomplete rows:',int(study.raw.isna().any(axis=1).sum()))


vdc1    float64
vdc2    float64
idc1    float64
idc2    float64
irr     float64
pvt     float64
f_nv      uint8
Raw shape: (1373798, 7) ; conventional missing cells: 0 ; paired zeros retained: 62
  vdc1   vdc2   idc1   idc2    irr    pvt  f_nv
0.7143 0.5550 0.0608 0.0073 1.3729 2.3816     0
0.6944 0.5427 0.0615 0.0064 1.3604 2.3816     0
0.7110 0.5583 0.0646 0.0067 1.5118 2.3883     0
0.6991 0.5465 0.0628 0.0067 1.5534 2.3920     0
0.7035 0.5553 0.0606 0.0076 1.4355 2.3920     0
0.7109 0.5496 0.0624 0.0075 1.3632 2.3913     0
0.7078 0.5465 0.0612 0.0074 1.2415 2.3913     0
0.7135 0.5596 0.0614 0.0079 1.3430 2.3913     0
0.7268 0.5633 0.0616 0.0070 1.5618 2.3937     0
0.7249 0.5596 0.0623 0.0075 1.3872 2.3937     0
               vdc1          vdc2          idc1          idc2           irr           pvt          f_nv
count  1.373798e+06  1.373798e+06  1.373798e+06  1.373798e+06  1.373798e+06  1.373798e+06  1.373798e+06
mean   1.351183e+02  1.347199e+02  2.276919e+00  2.000344e+00  2.576

## Dataset-size discrepancy

The irradiance-qualified population exactly matches literature counts, suggesting a possible population explanation. The source does not explicitly document that filtering operation; all original rows remain in the primary dataset.

In [3]:
qualified=study.raw.loc[study.raw.irr>=100]
assert len(qualified)==515958
assert qualified.f_nv.value_counts().sort_index().tolist()==[309253,5999,10371,6024,184311]
print('Raw count:',len(study.raw),'; irradiance-qualified count:',len(qualified))
print(qualified.f_nv.value_counts().sort_index().to_string())


Raw count: 1373798 ; irradiance-qualified count: 515958
f_nv
0    309253
1      5999
2     10371
3      6024
4    184311


## Ordered partitions and leakage architecture

The original 12/2/2 signal-block allocation and separator buffers remain. Blocks are not verified days/sessions. No timestamps are fabricated. All learned preprocessing statistics use TRAIN; validation/test are transform-only. Source order is not a guarantee of pristine prospective evaluation.

In [4]:
study.split_data()
assert [len(study.partitions[k]) for k in ['TRAIN','VALIDATION','TEST']]==[1006129,116977,146461]


                      interval        N                                                      index_digest
TRAIN             [0, 1006129]  1006129  86010f1f91f0653fd59a65ec55e3a3e3a7e905b6d292e7207099d8c4b58075f9
VALIDATION  [1056990, 1173967]   116977  8a7727a6679eb9bb52e6e40c689e4f4b054b9efe8f4cea32e4229af1b4be2328
TEST        [1227337, 1373798]   146461  e5dabd8b8c3ee5220c9ef21da1dbdfdbdd9aad99dfcc4c4353e698eebc753866


## Standalone Pipeline / ColumnTransformer

The original target-free transformations are extracted without model-loading dependencies. six_standard uses six raw predictors, median fallback, StandardScaler and six indicators. electrical_native uses four electrical predictors and four indicators. electrical_derived adds five parent-constrained electrical features. These are existing representations, not evidence that four channels are sufficient. Availability eligibility is tracked without deleting rows. No automatic statistical capping is performed.

In [5]:
study.model_ready()
assert all(list(pre.named_steps)==['quality','columns'] for pre in study.preprocessors.values())
assert study.stats['reference_cleaning']['removed_rows']==0
assert study.stats['reference_cleaning']['winsorized_cells']==0
print(study.stats['reference_cleaning'])


six_standard {'TRAIN': {'shape': [1006129, 12], 'eligible_rows': 1006129, 'target_rows': 1006129, 'fitted_again': False}, 'VALIDATION': {'shape': [116977, 12], 'eligible_rows': 116977, 'target_rows': 116977, 'fitted_again': False}, 'TEST': {'shape': [146461, 12], 'eligible_rows': 146461, 'target_rows': 146461, 'fitted_again': False}}
electrical_native {'TRAIN': {'shape': [1006129, 8], 'eligible_rows': 1006129, 'target_rows': 1006129, 'fitted_again': False}, 'VALIDATION': {'shape': [116977, 8], 'eligible_rows': 116977, 'target_rows': 116977, 'fitted_again': False}, 'TEST': {'shape': [146461, 8], 'eligible_rows': 146461, 'target_rows': 146461, 'fitted_again': False}}
electrical_derived {'TRAIN': {'shape': [1006129, 13], 'eligible_rows': 1006129, 'target_rows': 1006129, 'fitted_again': False}, 'VALIDATION': {'shape': [116977, 13], 'eligible_rows': 116977, 'target_rows': 116977, 'fitted_again': False}, 'TEST': {'shape': [146461, 13], 'eligible_rows': 146461, 'target_rows': 146461, 'fitted_

## Missingness and class distribution

All TRAIN labels remain at their natural prevalence. No oversampling or target encoding as a continuous variable is introduced.

In [6]:
counts=study.partitions['TRAIN'].f_nv.value_counts().sort_index()
assert counts.tolist()==[844805,1226,5695,1204,153199]
print(pd.DataFrame({'count':counts,'percent':100*counts/counts.sum()}).to_string())


       count    percent
f_nv                   
0     844805  83.965873
1       1226   0.121853
2       5695   0.566031
3       1204   0.119667
4     153199  15.226576


![TRAIN-only seeded missingness display; full export contains zero conventional missing cells.](../../figures/fig1_1_missingness_heatmap.png)

TRAIN-only seeded missingness display; full export contains zero conventional missing cells.

![All TRAIN operating states; no rebalancing. Normal represents approximately 83.966%.](../../figures/fig1_2_target_distribution.png)

All TRAIN operating states; no rebalancing. Normal represents approximately 83.966%.

## Outlier investigation and distribution evidence

TRAIN IQR fences use Q1 − 1.5 IQR and Q3 + 1.5 IQR. A distribution-qualified |Z| > 3 screen complements the IQR audit. Unusual measurements may carry real fault information, so statistical flags do not cause automatic deletion or winsorization. Median fallback is robust to skew; mean is outlier-sensitive, mode is unsuitable for these continuous measurements, and KNN/interpolation require extra assumptions. No conventional exported NaNs require imputation in this release. Short-unavailability eligibility is retained as a contingency only.

In [7]:
print(pd.read_csv(ROOT/'phase1_strategy/Training_Distributions.csv').to_string(index=False))
assert study.stats['reference_cleaning']['rows_before']==study.stats['reference_cleaning']['rows_after']==1006129


sensor       mean  median        std  skewness      q1       q3  iqr_lower  iqr_upper  iqr_flagged_count  iqr_flagged_pct  z_gt_3_count
  vdc1 137.108374 51.2504 136.995757  0.068287  1.0184 274.4720 -409.16200  684.65240                  0              0.0             0
  vdc2 136.923536 53.1038 137.283171  0.090920  0.9824 272.3780 -406.11100  679.47140                  0              0.0             0
  idc1   2.248129  0.1105   3.125250  1.004558  0.0648   4.9670   -7.28850   12.32030                  0              0.0             0
  idc2   1.976316  0.0432   2.988747  1.145043  0.0088   4.0011   -5.97965    9.98955                  0              0.0             0
   irr 257.928481  1.8733 349.018335  0.874462  1.2889 586.3810 -876.34925 1464.01915                  0              0.0             0
   pvt  23.116472 19.4501  14.349387  0.515723 12.5516  34.0299  -19.66585   66.24735                  0              0.0             0


![Raw versus reference-cleaned TRAIN distributions; original observations are retained.](../../figures/fig2_1_outlier_boxplots.png)

Raw versus reference-cleaned TRAIN distributions; original observations are retained.

## Predictor relationships and nominal-target association

Pearson correlation excludes f_nv. Five pairs exceed |r| ≥ 0.85. Scatterplots show a disclosed 20,000-position TRAIN sample; lines/correlations use full TRAIN. Mutual information uses a seeded proportional sample of 50,000 TRAIN observations, continuous predictors and nominal target, k=3. Association is not significance or causality; repeated measurements and ties limit interpretation.

In [8]:
study.associations()
assert len(study.high_pairs)==5
assert list(study.correlation.columns)==SENSORS
print(study.correlation.to_string())


Training |r| >= .85 pairs: [('vdc1', 'vdc2', 0.9891027602285769), ('idc1', 'idc2', 0.952832711587032), ('idc1', 'irr', 0.9794723761665458), ('idc1', 'pvt', 0.8622889138042289), ('idc2', 'irr', 0.9559821219702014)]
Training-sample mutual information (nats): {'vdc1': 0.1786151235282165, 'vdc2': 0.22061103822354466, 'idc1': 0.276227212137653, 'idc2': 0.27698098738977794, 'irr': 0.300591903247168, 'pvt': 0.11237539740848446}
          vdc1      vdc2      idc1      idc2       irr       pvt
vdc1  1.000000  0.989103  0.681183  0.640980  0.728635  0.645385
vdc2  0.989103  1.000000  0.691902  0.637323  0.739005  0.650481
idc1  0.681183  0.691902  1.000000  0.952833  0.979472  0.862289
idc2  0.640980  0.637323  0.952833  1.000000  0.955982  0.826632
irr   0.728635  0.739005  0.979472  0.955982  1.000000  0.842196
pvt   0.645385  0.650481  0.862289  0.826632  0.842196  1.000000


![TRAIN predictor-only correlation; no automatic sensor deletion.](../../figures/fig3_1_correlation_matrix.png)

TRAIN predictor-only correlation; no automatic sensor deletion.

![Five flagged pairs; display N=20,000, full-TRAIN descriptive lines.](../../figures/fig3_2_collinear_scatterplots.png)

Five flagged pairs; display N=20,000, full-TRAIN descriptive lines.

![Target Association / Mutual Information Feature Screening; not trained-model importance.](../../figures/fig3_3_target_importance.png)

Target Association / Mutual Information Feature Screening; not trained-model importance.

## Multicollinearity

The original TRAIN VIF audit is retained. Five values exceed 10; physical interpretation and regularization in later algorithms motivate retention rather than automatic removal. VIF/correlation alone does not establish redundant physical sensors.

In [9]:
vif=load('phase1_strategy/primary_audit.json')['training_vif']
assert sum(v>10 for v in vif.values())==5
print(pd.Series(vif,name='TRAIN VIF').to_string())


vdc1    52.168202
vdc2    56.345662
idc1    31.495610
idc2    15.780847
irr     39.292651
pvt      4.069798


## Scaling alternatives and exploratory PCA

StandardScaler supports later scale-sensitive regularized methods; native-scale representations support trees. RobustScaler and MinMaxScaler are diagnostic alternatives. Scaling does not make a skewed distribution Gaussian. No log transform is introduced. The original transformation image retains algorithm-rationale labels only, not model outcomes.

In [10]:
study.scaling_and_pca()
assert int(study.standard.n_samples_seen_)==1006129
assert study.pca.n_samples_==1006129
assert abs(study.pca.explained_variance_ratio_[:2].sum()-.9516766680256425)<1e-10


Exploratory PCA: {'population': 'TRAIN', 'N': 1006129, 'input_columns': ['vdc1', 'vdc2', 'idc1', 'idc2', 'irr', 'pvt'], 'scaled_with': 'Existing StandardScaler fitted on TRAIN only', 'explained_variance_ratio': [0.8253007587514459, 0.12637590927419656, 0.03575663896863875, 0.008328722852125044, 0.002756421992577408, 0.0014815481610162677], 'cumulative_explained_variance': [0.8253007587514459, 0.9516766680256425, 0.9874333069942813, 0.9957620298464064, 0.9985184518389838, 1.0], 'components_for_90pct': 2, 'components_for_95pct': 2, 'deployment_role': 'Exploratory diagnostic only; no physical sensor reduction without selecting physical channels.'}


![Original TRAIN scaling comparison; algorithm labels identify preprocessing rationale only.](../../figures/fig4_1_transformation_grid.png)

Original TRAIN scaling comparison; algorithm labels identify preprocessing rationale only.

![TRAIN-only exploratory PCA; two mathematical components retain approximately 95.17%, but require six physical inputs.](../../figures/fig5_1_pca_scree_plot.png)

TRAIN-only exploratory PCA; two mathematical components retain approximately 95.17%, but require six physical inputs.

## Output dimensions and physical feature semantics

Mathematical inputs differ from acquired channels: 12 = 6 raw + 6 indicators; 8 = 4 raw + 4 indicators; 13 = 4 raw + 5 derived + 4 indicators. Electrical RMS products are physical proxies, not independently measured instantaneous power. No physical sensor reduction is inferred.

In [11]:
rows=[]
for policy,rec in study.stats['model_ready'].items():
 for part,value in rec['partitions'].items():rows.append({'policy':policy,'partition':part,'rows':value['shape'][0],'mathematical_inputs':value['shape'][1],'physical_channels':len(rec['raw_channels'])})
assert [study.matrices[k]['TRAIN'].shape[1] for k in POLICIES]==[12,8,13]
print(pd.DataFrame(rows).to_string(index=False))
print(study.preprocessors['electrical_derived'].get_feature_names_out())


            policy  partition    rows  mathematical_inputs  physical_channels
      six_standard      TRAIN 1006129                   12                  6
      six_standard VALIDATION  116977                   12                  6
      six_standard       TEST  146461                   12                  6
 electrical_native      TRAIN 1006129                    8                  4
 electrical_native VALIDATION  116977                    8                  4
 electrical_native       TEST  146461                    8                  4
electrical_derived      TRAIN 1006129                   13                  4
electrical_derived VALIDATION  116977                   13                  4
electrical_derived       TEST  146461                   13                  4
['numeric__vdc1' 'numeric__vdc2' 'numeric__idc1' 'numeric__idc2'
 'numeric__P1' 'numeric__P2' 'numeric__P_total' 'numeric__V_diff'
 'numeric__I_diff' 'missing__missingindicator_vdc1'
 'missing__missingindicator_vdc2' 'mi

## Train-only fit ledger and reference consistency

All compact scientific reference values are compared with the executed preprocessing. The original images are verified rather than regenerated. Arrays remain in memory. The current [strategy-versus-reality table](../compliance/Strategy_vs_Reality.md) records preprocessing only.

In [12]:
assert all(x['population']=='TRAIN' and x['N']==1006129 for x in study.stats['fit_ledger'])
from src.preprocessing.validation import validate_study
validate_study(study)
print(pd.DataFrame(study.stats['fit_ledger']).drop(columns=['index_digest']).to_string(index=False))


                                      component population       N                       raw_channels diagnostic_only
      six_standard median/scaler/feature schema      TRAIN 1006129 [vdc1, vdc2, idc1, idc2, irr, pvt]             NaN
 electrical_native median/scaler/feature schema      TRAIN 1006129           [vdc1, vdc2, idc1, idc2]             NaN
electrical_derived median/scaler/feature schema      TRAIN 1006129           [vdc1, vdc2, idc1, idc2]             NaN
                       RobustScaler alternative      TRAIN 1006129                                NaN            True
                       MinMaxScaler alternative      TRAIN 1006129                                NaN            True
                Standard-scaled exploratory PCA      TRAIN 1006129                                NaN            True


## Measured stage dimensions and empirical reflection
The raw matrix is 1,373,798 ? 7 including the nominal target. Predictor matrices begin with six columns. Each quality, median, indicator, physical-feature and scaling stage is measured below for all three ordered partitions. Indicators form a parallel ColumnTransformer branch; their logical combination with median-imputed raw values is distinguished from the actual final derived/scaled numeric branch. PCA is a TRAIN-only six-input diagnostic, not a selected reduction in physical channels. Statistical diagnostics use TRAIN only; original Phase 1 values are checked before new kurtosis/eigenvalue evidence is recorded.

In [13]:
study.figures=load('phase2_implementation/compliance/Diagnostic_Inventory.json')
assert len(study.figures)==8
for item in study.figures:assert sha(ROOT/item['file'])==item['sha256']
from src.preprocessing.empirical_audit import empirical_audit
dimensions, distributions, scaling_evidence, pca_evidence=empirical_audit(study)
print('Measured N x P at every preprocessing stage:')
print(dimensions.to_string(index=False))
print('TRAIN distribution and outlier evidence:')
print(distributions.to_string(index=False))
print('TRAIN PCA eigenvalues and ratios:')
print(pca_evidence.to_string(index=False))
assert all(study.stats['model_ready'][k]['partitions']['TRAIN']['shape'][1]==n for k,n in [('six_standard',12),('electrical_native',8),('electrical_derived',13)])
study.save_evidence()
print('Preprocessing complete; eight original image hashes verified; no downstream evaluation.')


Measured N x P at every preprocessing stage:
            policy  partition  input_N  original_predictors  retained_physical_channels  post_quality_P  post_median_numeric_P  indicator_P  post_imputation_plus_indicators_P  post_feature_numeric_P  post_scaling_numeric_P  final_design_P  post_feature_plus_indicators_P  PCA_diagnostic_input_P  PCA_diagnostic_all_components_P  PCA_95pct_view_P input_predictor_NxP post_quality_NxP post_imputation_indicators_NxP post_feature_numeric_NxP post_scaling_numeric_NxP final_design_NxP
      six_standard      TRAIN  1006129                    6                           6               6                      6            6                                 12                       6                       6              12                              12                     6.0                              6.0               2.0         1006129 x 6      1006129 x 6                   1006129 x 12              1006129 x 6              1006129 x 6     100612